<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/01-model-apis/02-tool-calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool Calling

**Goal:** Run function/tool calling end to end, including the error paths.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client — the only dependency this notebook needs.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

## Why tools, and the round trip

A model knows only what was in its training data. Ask it today's weather in Tokyo or `(17.5% of 2340)` and it will *guess* — confidently, often wrong. **Tool calling** closes that gap: you hand the model a set of functions it can ask you to run — hit a weather API, do exact arithmetic, query your database — so live data and real computation flow into the answer instead of being hallucinated. This is the mechanism behind every "agent": the model's reach into the world is exactly the tools you give it.

The key mental model: **tool calling is a protocol, not magic. The model never executes anything** — it emits a *request* to call a tool, and your code does the rest:

1. You send `tools=[...]` with the user message.
2. The model replies with `finish_reason == 'tool_calls'` and one or more tool calls on `message.tool_calls` (tool name + JSON arguments string + an `id`).
3. **You** execute the function, then send back a `role: "tool"` message with the result referencing that `id`.
4. The model continues — possibly calling more tools — until it answers in plain text (`finish_reason == 'stop'`).

The API is stateless: every step resends the whole conversation. Two tools for this notebook — a fake `get_weather` (canned data, so the notebook runs anywhere) and a real `calculate` that safely evaluates arithmetic.


In [ ]:
import ast
import json
import operator

TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': (
                'Get the current weather for a city. Call this whenever the user asks about '
                'weather, temperature, or outdoor conditions in a specific place.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {'type': 'string', 'description': 'City name, e.g. "Tokyo"'},
                    'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']},
                },
                'required': ['city'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'calculate',
            'description': (
                'Evaluate an arithmetic expression exactly. Call this for any math beyond '
                'trivial mental arithmetic instead of computing in your head. '
                'Supports + - * / ** and parentheses on numbers only.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'expression': {'type': 'string', 'description': 'e.g. "(17.5 * 12) / 3"'},
                },
                'required': ['expression'],
            },
        },
    },
]


# --- implementations ---

FAKE_WEATHER = {
    'tokyo': {'temp_c': 31, 'conditions': 'humid, partly cloudy'},
    'paris': {'temp_c': 24, 'conditions': 'clear'},
    'london': {'temp_c': 18, 'conditions': 'light rain'},
}


def get_weather(city, unit='celsius'):
    data = FAKE_WEATHER.get(city.lower())
    if data is None:
        raise KeyError(f'no weather data for {city!r}')
    temp = data['temp_c'] if unit == 'celsius' else round(data['temp_c'] * 9 / 5 + 32)
    return f"{city}: {temp} degrees {unit}, {data['conditions']}"


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def calculate(expression):
    """Evaluate arithmetic via the AST — never eval() model-provided strings."""
    def ev(node):
        if isinstance(node, ast.Expression):
            return ev(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError(f'unsupported expression: {expression!r}')
    return ev(ast.parse(expression, mode='eval'))


HANDLERS = {'get_weather': get_weather, 'calculate': calculate}


## Step 1: see the raw tool request

Before building the loop, look at what the model actually sends back. On the OpenAI-compatible interface each entry in `message.tool_calls` has `tc.function.name`, `tc.function.arguments` (a JSON **string** you parse with `json.loads`, not a dict), and `tc.id` — the handle you echo back with the result.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=500,
    tools=TOOLS,
    messages=[{'role': 'user', 'content': 'What is the weather in Tokyo right now?'}],
)

print('finish_reason:', response.choices[0].finish_reason)
msg = response.choices[0].message
if msg.content:
    print('text:', msg.content)
if msg.tool_calls:
    for tc in msg.tool_calls:
        print(f'tool_call: {tc.function.name}({tc.function.arguments})  id={tc.id}')


Run it and note `finish_reason` is `tool_calls` — the model has paused mid-turn, waiting for you. There may also be a short text chunk before the tool call; that's normal.

## Step 2: the full loop

Now close the circuit. Three rules most tutorials get right, and two they skip:

- Append the assistant's **entire** message object before the results — the `tool_calls` list must stay in history.
- Return **all** tool results as separate `role: "tool"` messages, each matched by `tool_call_id`.
- Cap the loop. A model that keeps calling tools forever should hit your ceiling, not your credit card.

Skipped by tutorials: exceptions (wrap the handler; return an error string so the model can recover gracefully) and parallel calls (one response can contain *several* tool calls — handle all of them, not just the first).


In [ ]:
def execute_tool(tool_call):
    """Run one tool call; returns a tool message (error string or result)."""
    try:
        handler = HANDLERS[tool_call.function.name]
        args = json.loads(tool_call.function.arguments)
        result = handler(**args)
        content = str(result)
        is_error = False
    except Exception as e:
        content = f'{type(e).__name__}: {e}'
        is_error = True
    return {'role': 'tool', 'tool_call_id': tool_call.id, 'content': content}, is_error


def run_agent(user_message, max_turns=5, verbose=True):
    messages = [{'role': 'user', 'content': user_message}]
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL, max_tokens=600, tools=TOOLS, messages=messages,
        )
        choice = response.choices[0]
        if choice.finish_reason != 'tool_calls':
            return choice.message.content or ''

        # Append assistant message (must keep tool_calls intact for history)
        messages.append(choice.message)

        # Execute ALL tool calls and append each as a separate tool message
        for tc in choice.message.tool_calls:
            tool_msg, is_error = execute_tool(tc)
            if verbose:
                flag = ' [ERROR]' if is_error else ''
                print(f"  turn {turn}: {tc.function.name}({tc.function.arguments}) "
                      f"-> {tool_msg['content']}{flag}")
            messages.append(tool_msg)

    raise RuntimeError(f'agent did not finish within {max_turns} turns')


print(run_agent('What is the weather in Tokyo, and what is 17.5% of 2340?'))


## Error path 1: the tool raises

Ask about a city our fake weather source doesn't know. `get_weather` raises `KeyError`, `execute_tool` catches it and returns a `role: "tool"` message carrying the error text (with `is_error=True` so the trace can flag it), and the model gets to decide what to do — apologize, ask for clarification, or try something else. Run it and note the model produces a graceful answer instead of the loop crashing.

In [ ]:
print(run_agent('What is the weather in Reykjavik?'))


## Error path 2: invalid arguments

The model can also call *your* tool wrong — an expression your calculator doesn't support, a missing field, a wrong type. Two layers of defense, and they're complementary, not either/or:

| Defense | Catches | Misses | Support | Use it for |
|---|---|---|---|---|
| **`'strict': True` schema** (API-side) | wrong shape: missing required field, wrong type, unknown key | *semantically* bad values that fit the schema | uneven across Groq models — treat as a bonus | cheap first filter on structure |
| **Your handler validates** (code) | anything: undefined variables, out-of-range values, business rules | nothing — it's your code | always available | the real guarantee; keep it regardless |

- **Your handler validates.** `calculate` rejects anything that isn't pure arithmetic (run the cell below: the model relays the tool's complaint or retries with a fixed expression).
- **The API can validate for you.** Some providers support `'strict': True` on a tool definition (with `additionalProperties: False` and a `required` list) so the arguments are guaranteed to match the schema before they reach your code. Support is uneven across Groq models, so treat it as a bonus, not a guarantee. And schema validation can't catch a value that *fits the shape but is wrong* (an expression that parses but references an undefined variable), so the handler check stays either way.

In [ ]:
# 'x' is not defined, so the AST evaluator rejects the expression.
# Run this and watch the error round trip: is_error -> model recovers or explains.
print(run_agent('Use the calculate tool to evaluate "x + 2" where x is 5.'))


## Parallel tool calls

By default the model may request several tools in one response — a single assistant message whose `tool_calls` list holds multiple entries. Our loop already handles this: it iterates every entry and appends one `role: "tool"` message per call, each matched back by `tool_call_id`. That completeness is load-bearing — every tool call in the assistant message must get a matching result before the next request, or the API rejects the conversation as malformed.

In [ ]:
# A question that invites three tool calls at once. Watch how many happen per turn.
print(run_agent(
    'Compare the current weather in Tokyo, Paris, and London. '
    'Then tell me the average of their temperatures in celsius (use calculate).'
))


Run it and note the turn numbers in the trace: the three `get_weather` calls typically land in the *same* turn (parallel), then `calculate` follows in the next one because it depends on their results. The model figured out that dependency ordering on its own.

## Tool descriptions are prompts

The single highest-leverage line in a tool definition is the `description`. The model decides *whether* and *how* to call your tool almost entirely from it. Vague descriptions produce wrong calls:

- `"Gets weather"` — under-triggers (model answers from memory) and invites wrong arguments (country instead of city).
- `"Get the current weather for a city. Call this whenever the user asks about weather... "` — states *when* to call it, not just what it does. Trigger conditions in the description measurably improve call rates on current models, which are conservative about reaching for tools.

Same discipline as API docs: describe each parameter, use `enum` for closed sets, mark only truly required fields as required. If the model keeps misusing a tool, fix the description before you fix the prompt.


## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Parse `tc.function.arguments` with `json.loads`; append the assistant message *and* one `role:"tool"` result per call | Forget a tool result, or mutate the assistant message — the API rejects the next request |
| Wrap the handler; return a *teaching* error string the model can recover from | Let a handler exception crash the loop |
| Handle **all** tool calls in a response (parallel calls are normal) | Handle only `tool_calls[0]` and drop the rest |
| Cap the loop with `max_turns` | Unbounded tool loop against a paid key |
| Write descriptions that state *when* to call; validate args in code | Vague descriptions; trust `strict:true` alone (uneven support) |

(The full stack-wide list lives in [docs/best-practices-and-anti-patterns.md](https://github.com/calmrocks/ai-engineer-notebooks/blob/main/docs/best-practices-and-anti-patterns.md).)

## Exercises

1. Add a third tool `convert_currency(amount, from_ccy, to_ccy)` backed by a hardcoded rate table, then ask a question that needs weather + currency + math in one request. Check the trace for parallelism.
2. Make `get_weather` fail randomly 50% of the time with a `TimeoutError`, and extend `execute_tool` to retry once before returning `is_error`. Compare transcripts with and without the retry.
3. Add `'strict': True` (plus `additionalProperties: False` and full `required` lists) to both tool definitions, then try to provoke a malformed call. Verify bad shapes no longer reach your handlers.
4. Rewrite the `calculate` description to be maximally vague ("does math stuff") and re-run the percentage question 3 times. Count how often the model computes in its head instead of calling the tool.
